# Paper PoRT Recreated Scale Run

This notebook scales the recreated PoRT path after notebook 19 passed its best-classifier smoke matrix. It reuses the same post-judge classifier logic: `answer_only` TF-IDF/logistic classifier plus generated-letter expansion to full choice text before classification.

Default mode is a medium run with `PORT_MAX_SAMPLES=32` per variant/domain job. Set `PORT_MAX_SAMPLES=-1` to run the full selected datasets after the medium run is stable.

This is still a recreated-artifact run, not an official paper-checkpoint metric run.


In [1]:
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys

REPO_URL = 'https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git'
REPO_DIR_NAME = 'PoRT_LLM_Unlearning-Experiment'
IS_KAGGLE = Path('/kaggle/working').exists()


def has_project_layout(path):
    path = Path(path)
    return (path / 'PoRT_pipeline' / 'WMDP' / 'port_pipeline_wmdp.py').exists() and (path / 'dataset').exists()


def clone_or_use_project():
    if IS_KAGGLE:
        target = Path('/kaggle/working') / REPO_DIR_NAME
        if has_project_layout(target):
            print(f'Using existing cloned repository: {target}')
            subprocess.check_call(['git', '-C', str(target), 'pull', '--ff-only'])
            return target.resolve()
        if target.exists():
            raise RuntimeError(f'{target} exists but does not look like this repo.')
        print(f'Cloning {REPO_URL} into {target}')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
        return target.resolve()

    local_root = Path.cwd().resolve()
    if has_project_layout(local_root):
        return local_root
    target = local_root / REPO_DIR_NAME
    if has_project_layout(target):
        return target.resolve()
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
    return target.resolve()


PROJECT_ROOT = clone_or_use_project()
os.environ['PORT_PROJECT_ROOT'] = str(PROJECT_ROOT)
commit_sha = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print(f'Project root: {PROJECT_ROOT}')
print(f'Commit: {commit_sha}')


Cloning https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git into /kaggle/working/PoRT_LLM_Unlearning-Experiment


Cloning into '/kaggle/working/PoRT_LLM_Unlearning-Experiment'...


Project root: /kaggle/working/PoRT_LLM_Unlearning-Experiment
Commit: c51a91213162e7f7f6ee1d059d842ef849b2dab6


In [2]:
required_packages = {
    'datasets': 'datasets>=2.10.1',
    'joblib': 'joblib',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'pyarrow': 'pyarrow>=10',
    'safetensors': 'safetensors',
    'sklearn': 'scikit-learn',
    'torch': 'torch',
    'transformers': 'transformers>=4.38.0',
    'sentencepiece': 'sentencepiece',
    'yaml': 'pyyaml',
    'tqdm': 'tqdm',
}

missing_packages = []
for module_name, package_spec in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing_packages.append(package_spec)

if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('Required packages are already available.')


Required packages are already available.


## Runtime Config

Defaults are chosen for a first scale run:

- `PORT_WMDP_VARIANTS=original,noise_prefix,composite`
- `PORT_WMDP_DOMAINS=bio,chem,cyber`
- `PORT_MAX_SAMPLES=32`
- `PORT_RESUME_EXISTING=true`
- `PORT_CLASSIFIER_CONF_THRESHOLD=0.70`

Useful overrides before running this notebook:

- `PORT_MAX_SAMPLES=64` for a larger medium run.
- `PORT_MAX_SAMPLES=-1` for the full selected datasets.
- `PORT_RECREATED_ARTIFACT_ZIP_URL=https://.../paper_port_recreated_artifacts_bootstrap.zip` to reuse a hosted artifact zip instead of bootstrapping in the notebook.


In [3]:
os.environ.setdefault('PORT_ARTIFACT_MODE', 'recreated')
os.environ.setdefault('PORT_MAX_SAMPLES', '32')
os.environ.setdefault('PORT_RESUME_EXISTING', 'true')
os.environ.setdefault('PORT_FAIL_FAST', 'true')
os.environ.setdefault('PORT_BEST_CLASSIFIER_SAMPLES_PER_DOMAIN', '256')
os.environ.setdefault('PORT_BEST_CLASSIFIER_WRONG_ANSWERS_PER_QUESTION', '3')
os.environ.setdefault('PORT_BEST_CLASSIFIER_FEATURE_SET', 'answer_only')
os.environ.setdefault('PORT_BEST_CLASSIFIER_MAX_FEATURES', '50000')

runtime_keys = [
    'PORT_ARTIFACT_MODE',
    'PORT_RUN_NAME',
    'PORT_WMDP_VARIANTS',
    'PORT_WMDP_DOMAINS',
    'PORT_MAX_SAMPLES',
    'PORT_RESUME_EXISTING',
    'PORT_FAIL_FAST',
    'PORT_CLASSIFIER_CONF_THRESHOLD',
    'PORT_BEST_CLASSIFIER_FEATURE_SET',
    'PORT_RECREATED_ARTIFACT_DIR',
    'PORT_RECREATED_ARTIFACT_ZIP_URL',
    'PORT_RECREATED_ARTIFACT_ZIP_PATH',
]
print(json.dumps({key: os.environ.get(key) for key in runtime_keys}, indent=2))


{
  "PORT_ARTIFACT_MODE": "recreated",
  "PORT_RUN_NAME": null,
  "PORT_WMDP_VARIANTS": null,
  "PORT_WMDP_DOMAINS": null,
  "PORT_MAX_SAMPLES": "32",
  "PORT_RESUME_EXISTING": "true",
  "PORT_FAIL_FAST": "true",
  "PORT_CLASSIFIER_CONF_THRESHOLD": null,
  "PORT_BEST_CLASSIFIER_FEATURE_SET": "answer_only",
  "PORT_RECREATED_ARTIFACT_DIR": null,
  "PORT_RECREATED_ARTIFACT_ZIP_URL": null,
  "PORT_RECREATED_ARTIFACT_ZIP_PATH": null
}


In [4]:
import importlib.util

runner_path = PROJECT_ROOT / 'notebooks' / 'common' / 'port_recreated_scale_run.py'
if not runner_path.exists():
    raise FileNotFoundError(runner_path)

common_dir = str(runner_path.parent)
if common_dir not in sys.path:
    sys.path.insert(0, common_dir)

spec = importlib.util.spec_from_file_location('port_recreated_scale_run', runner_path)
port_recreated_scale_run = importlib.util.module_from_spec(spec)
spec.loader.exec_module(port_recreated_scale_run)

result = port_recreated_scale_run.run(
    project_root=PROJECT_ROOT,
    is_kaggle=IS_KAGGLE,
    commit_sha=commit_sha,
)
print(json.dumps(result, indent=2, default=str))

run_dir = Path(result['run_dir'])
for artifact_name in [
    'artifact_audit.json',
    'run_config.json',
    'summary.json',
    'matrix_summary.csv',
    'summary_by_variant_domain.csv',
    'all_predictions.csv',
    'failed_jobs.json',
]:
    artifact_path = run_dir / artifact_name
    print(f'{artifact_name}: {artifact_path.exists()} {artifact_path}')


{
  "artifact_mode": "recreated",
  "seed": 1234,
  "target_model_hub_name": "microsoft/phi-1_5",
  "target_model_path": "microsoft/phi-1_5",
  "model_name": "phi-1_5",
  "torch_dtype": "float16",
  "device": "cuda:0",
  "wmdp_variants": [
    "original",
    "noise_prefix",
    "composite"
  ],
  "wmdp_domains": [
    "bio",
    "chem",
    "cyber"
  ],
  "max_samples": 32,
  "batch_size": 1,
  "icl_example_k": 3,
  "classifier_conf_threshold": 0.7,
  "prefix_prompt_max_length": 1024,
  "prefix_max_new_tokens": 128,
  "answer_prompt_max_length": 1536,
  "answer_max_new_tokens": 32,
  "recreated_artifact_dir_env": null,
  "recreated_artifact_zip_url": null,
  "recreated_artifact_zip_path": null,
  "bootstrap_recreated_if_missing": true,
  "bootstrap_train_t5": true,
  "t5_base_model": "google/flan-t5-small",
  "t5_epochs": 3,
  "t5_batch_size": 4,
  "t5_lr": 5e-05,
  "t5_max_input_length": 512,
  "t5_max_target_length": 512,
  "classifier_samples_per_split": 64,
  "classifier_max_featu

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.84G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/341 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]


=== Job 1/9: original/bio, rows=32, prompt_source=question_plus_choices ===
{
  "variant": "original",
  "domain": "bio",
  "wmdp_set": "wmdp-bio",
  "prompt_source": "question_plus_choices",
  "accuracy": 0.25,
  "valid_predictions_rate": 1.0,
  "rethink_count": 26,
  "rethink_rate": 0.8125,
  "postjudge_positive_rate": 0.34375,
  "postjudge_avg_confidence": 0.6611403487622738,
  "model_load_seconds": 28.45398943700002,
  "run_seconds": 153.632037783,
  "rows": 32,
  "output_dir": "/kaggle/working/paper_port_wmdp_recreated_scale_run_phi-1_5/port_outputs/phi-1_5/original/wmdp-bio",
  "resume_status": "computed"
}

=== Job 2/9: original/chem, rows=32, prompt_source=question_plus_choices ===
{
  "variant": "original",
  "domain": "chem",
  "wmdp_set": "wmdp-chem",
  "prompt_source": "question_plus_choices",
  "accuracy": 0.21875,
  "valid_predictions_rate": 1.0,
  "rethink_count": 18,
  "rethink_rate": 0.5625,
  "postjudge_positive_rate": 0.25,
  "postjudge_avg_confidence": 0.6839660927